# Scrapping data about journals from https://www.scimagojr.com/journalsearch

In [3]:
import re
import html
import urllib.request
import urllib.parse
from bs4 import BeautifulSoup
import time
import json
import os
import ast # to read All_jorunals
import threading
from helper_functions import *

ModuleNotFoundError: No module named 'helper_functions'

## Scrapping

### Get a List of All Journals

In [10]:
#to read it back

with open('All journals.txt','r') as f:
   All_journals = ast.literal_eval(f.read())

In [11]:
All_journals=list(All_journals)
len(All_journals)

40994

In [4]:
cou=0
for i in Parallel_journals:
    cou+=len(i)
print(cou)

NameError: name 'Parallel_journals' is not defined

In [26]:
Parallel_journals=[]
Cores=2
for i in range(Cores):
    index_=len(All_journals)//Cores
    if i==Cores-1:
        Parallel_journals.append(All_journals[index_*i:])
    else:
        Parallel_journals.append(All_journals[index_*i:index_*(i+1)])

0
1


## PARALLEL

In [12]:
import re
import html
import urllib.request
import urllib.parse
from bs4 import BeautifulSoup
import time
import json
import os
import ast # to read All_jorunals
import threading
#from helper_functions import *

def In_Parallel(All_journal):
    
    counter=0
    s=1
    min_10=time.time()
    for j in All_journal:
        #if os.path.isfile(os.path.join("All Scrapped Data", str(j)+'.json')):
         #   counter+=1
         #   continue
        Scrap(str(j))
        counter+=1
    
        
        middle_way=time.time()-min_10
        
        if middle_way/60>10:
            print(str(s)+" 10 Minutes passed. Index= ", counter)
            s+=1
            min_10=time.time()
            
def Scrap(journal):
    request = urllib.request.Request("https://www.scimagojr.com/journalsearch.php?q="+journal+"&tip=sid", headers=headers)
    
    raw_data = urllib.request.urlopen(request).read()
    
    data = raw_data.decode("utf-8")
    
    soup = BeautifulSoup(data, "html.parser")
    
    #Write out json file for each journal:
    Json_file={}
    #self_cites:
    search="the total number of citations and journal"
    soup1=findInSoup(search, soup) #search for the Table
    key="Self Cites"
    not_key="Total Cites"
    self_cites, total_cites=ParseSoupTable(soup1,key, not_key)
    Json_file["Self Cites"]=self_cites
    
    search="Ratio of a journal's items, grouped in three years windows"
    soup2=findInSoup(search, soup)
    key="Cited documents"
    not_key="Uncited documents"
    cited_doc, uncited_doc=ParseSoupTable(soup2, key, not_key)
    Json_file["Cited documents"]=cited_doc
    Json_file["Uncited documents"]=uncited_doc

    search="Not every article in a journal is considered primary research and therefore"
    soup2=findInSoup(search, soup)
    key="Citable documents"
    not_key="Non-citable documents"
    citable_doc, noncitable_doc=ParseSoupTable(soup2, key, not_key)
    Json_file["Citable documents"]=citable_doc
    
    search="It represents the potential financial worth of a journal. It is obtained by multiplying"
    soup2=findInSoup(search, soup)
    key="Est. value (USD)"
    not_key=""
    fin_value=ParseSoupTable(soup2, key, not_key, finance=True)
    Json_file["Est. value (USD)"]=fin_value
    with open(os.path.join("All Scrapped Data", journal+'.json'), 'w') as fp:
        json.dump(Json_file, fp, indent=4)
def findInSoup(search, soup):
    for i in soup.find_all(class_="cellslide"):
        try:
            #print(i.p.text)
            if search in i.p.text:
                return i
        except:
            continue
def ParseSoupTable(soup1, key, not_key, finance=False):
    """
    Parse tables
    """
    key_dict={"Year":[], key:[]}
    #adding description
    #key_dict["Description"]= soup1.p.text.strip()
    
    if not finance:
        not_key_dict={"Year":[], not_key:[]}
        for i in soup1.table.find_all("tr")[1:]:
            #loop through every entry:
            flag=False
            for j in range(3):
                td=i.find_all("td")
                if j==0 and key in td[j].text:
                    flag=True
                    continue
                if flag:
                    if j==1:
                        key_dict["Year"].append(td[j].text)
                    else:
                        key_dict[key].append(td[j].text)
                elif j!=0:
                    if j==1:
                        not_key_dict["Year"].append(td[j].text)
                    else:
                        not_key_dict[not_key].append(td[j].text)
        return [key_dict, not_key_dict]
    else:
        for i in soup1.table.find_all("tr")[1:]:
            #loop through every entry:
            for j in range(2):
                td=i.find_all("td")
                if j==0:
                    key_dict["Year"].append(td[j].text)
                else:
                    key_dict[key].append(td[j].text)

        return key_dict

start=time.time()

user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36'

headers={'User-Agent':user_agent} 

Cores=13
#Run in Parallel:
Parallel_journals=[]
for i in range(Cores):
    index_=len(All_journals)//Cores
    if i==Cores-1:
        Parallel_journals.append(All_journals[index_*i:])
    else:
        Parallel_journals.append(All_journals[index_*i:index_*(i+1)])

threads=[]
for i in range(Cores):
    threads.append(threading.Thread(target=In_Parallel, args=(Parallel_journals[i], )))

for i in range(Cores):
    threads[i].start()
for i in range(Cores):
    threads[i].join()


end=time.time()
print("Time ", end-start)

1 10 Minutes passed. Index=  335
1 10 Minutes passed. Index=  257
1 10 Minutes passed. Index=  276
1 10 Minutes passed. Index=  339
1 10 Minutes passed. Index=  242
1 10 Minutes passed. Index=  332
1 10 Minutes passed. Index=  243
1 10 Minutes passed. Index=  282
1 10 Minutes passed. Index=  280
1 10 Minutes passed. Index=  274
1 10 Minutes passed. Index=  337
1 10 Minutes passed. Index=  247
1 10 Minutes passed. Index=  284
1 10 Minutes passed. Index= 1 10 Minutes passed. Index=  582
 687
1 10 Minutes passed. Index=  662
1 10 Minutes passed. Index=  570
1 10 Minutes passed. Index=  483
1 10 Minutes passed. Index=  564
1 10 Minutes passed. Index=  652
1 10 Minutes passed. Index=  677
1 10 Minutes passed. Index=  485
1 10 Minutes passed. Index=  562
1 10 Minutes passed. Index=  566
1 10 Minutes passed. Index=  487
1 10 Minutes passed. Index=  567
1 10 Minutes passed. Index=  731
1 10 Minutes passed. Index=  997
1 10 Minutes passed. Index=  850
1 10 Minutes passed. Index=  1033
1 10 Minu

## Helper Functions

In [38]:
def In_Parallel(All_journal):
    
    counter=0
    min_10=time.time()
    for j in All_journal:
        if os.path.isfile(os.path.join("All Scrapped Data", str(j)+'.json')):
            counter+=1
            continue
        Scrap(str(j))
        counter+=1
    
        
        middle_way=time.time()-min_10
        if middle_way/60>10:
            print("10 Minutes passed. Index= ", counter)
            min_10=time.time()

In [1]:
def In_Parallel(All_journal):
    
    counter=0
    min_10=time.time()
    for j in All_journal:
        if os.path.isfile(os.path.join("All Scrapped Data", str(j)+'.json')):
            counter+=1
            continue
        Scrap(str(j))
        counter+=1
    
        
        middle_way=time.time()-min_10
        if middle_way/60>10:
            print("10 Minutes passed. Index= ", counter)
            min_10=time.time()
            
def Scrap(journal):
    request = urllib.request.Request("https://www.scimagojr.com/journalsearch.php?q="+journal+"&tip=sid", headers=headers)
    
    raw_data = urllib.request.urlopen(request).read()
    
    data = raw_data.decode("utf-8")
    
    soup = BeautifulSoup(data, "html.parser")
    
    #Write out json file for each journal:
    Json_file={}
    #self_cites:
    search="the total number of citations and journal"
    soup1=findInSoup(search, soup) #search for the Table
    key="Self Cites"
    not_key="Total Cites"
    self_cites, total_cites=ParseSoupTable(soup1,key, not_key)
    Json_file["Self Cites"]=self_cites
    
    search="Ratio of a journal's items, grouped in three years windows"
    soup2=findInSoup(search, soup)
    key="Cited documents"
    not_key="Uncited documents"
    cited_doc, uncited_doc=ParseSoupTable(soup2, key, not_key)
    Json_file["Cited documents"]=cited_doc
    Json_file["Uncited documents"]=uncited_doc

    search="Not every article in a journal is considered primary research and therefore"
    soup2=findInSoup(search, soup)
    key="Citable documents"
    not_key="Non-citable documents"
    citable_doc, noncitable_doc=ParseSoupTable(soup2, key, not_key)
    Json_file["Citable documents"]=citable_doc
    
    search="It represents the potential financial worth of a journal. It is obtained by multiplying"
    soup2=findInSoup(search, soup)
    key="Est. value (USD)"
    not_key=""
    fin_value=ParseSoupTable(soup2, key, not_key, finance=True)
    Json_file["Est. value (USD)"]=fin_value
    with open(os.path.join("All Scrapped Data", journal+'.json'), 'w') as fp:
        json.dump(Json_file, fp, indent=4)
def findInSoup(search, soup):
    for i in soup.find_all(class_="cellslide"):
        try:
            #print(i.p.text)
            if search in i.p.text:
                return i
        except:
            continue
def ParseSoupTable(soup1, key, not_key, finance=False):
    """
    Parse tables
    """
    key_dict={"Year":[], key:[]}
    #adding description
    #key_dict["Description"]= soup1.p.text.strip()
    
    if not finance:
        not_key_dict={"Year":[], not_key:[]}
        for i in soup1.table.find_all("tr")[1:]:
            #loop through every entry:
            flag=False
            for j in range(3):
                td=i.find_all("td")
                if j==0 and key in td[j].text:
                    flag=True
                    continue
                if flag:
                    if j==1:
                        key_dict["Year"].append(td[j].text)
                    else:
                        key_dict[key].append(td[j].text)
                elif j!=0:
                    if j==1:
                        not_key_dict["Year"].append(td[j].text)
                    else:
                        not_key_dict[not_key].append(td[j].text)
        return [key_dict, not_key_dict]
    else:
        for i in soup1.table.find_all("tr")[1:]:
            #loop through every entry:
            for j in range(2):
                td=i.find_all("td")
                if j==0:
                    key_dict["Year"].append(td[j].text)
                else:
                    key_dict[key].append(td[j].text)

        return key_dict

### Citable documents

In [8]:
request = urllib.request.Request("https://www.scimagojr.com/journalsearch.php?q="+"28773"+"&tip=sid")

raw_data = urllib.request.urlopen(request).read()

data = raw_data.decode("utf-8")

soup = BeautifulSoup(data, "html.parser")
search="Not every article in a journal is considered primary research and therefore"
soup2=findInSoup(search, soup)
key="Citable documents"
not_key="Non-citable documents"
citable_doc, noncitable_doc=ParseSoupTable(soup2, key, not_key)
Json_file["Citable documents"]=citable_doc

In [9]:
citable_doc

{'Year': ['1999',
  '2000',
  '2001',
  '2002',
  '2003',
  '2004',
  '2005',
  '2006',
  '2007',
  '2008',
  '2009',
  '2010',
  '2011',
  '2012',
  '2013',
  '2014',
  '2015',
  '2016',
  '2017',
  '2018',
  '2019',
  '2020',
  '2021',
  '2022',
  '2023',
  '2024'],
 'Citable documents': ['74',
  '76',
  '72',
  '72',
  '67',
  '70',
  '64',
  '68',
  '83',
  '99',
  '106',
  '101',
  '92',
  '95',
  '94',
  '102',
  '106',
  '109',
  '107',
  '102',
  '88',
  '79',
  '78',
  '85',
  '89',
  '81']}

## self cites

In [157]:
search="the total number of citations and journal"
soup1=findInSoup(search) #search for the Table
key="Self Cites"
not_key="Total Cites"
self_cites, total_cites=ParseSoupTable(soup1,key, not_key)

In [158]:
self_cites

{'Year': ['1999',
  '2000',
  '2001',
  '2002',
  '2003',
  '2004',
  '2005',
  '2006',
  '2007',
  '2008',
  '2009',
  '2010',
  '2011',
  '2012',
  '2013',
  '2014',
  '2015',
  '2016',
  '2017',
  '2018',
  '2019',
  '2020',
  '2021',
  '2022',
  '2023',
  '2024'],
 'Self Cites': ['5',
  '12',
  '6',
  '3',
  '16',
  '21',
  '15',
  '14',
  '18',
  '18',
  '20',
  '28',
  '12',
  '15',
  '20',
  '18',
  '29',
  '48',
  '27',
  '36',
  '48',
  '29',
  '36',
  '34',
  '25',
  '27']}

## Cited documents uncited

In [145]:
search="Ratio of a journal's items, grouped in three years windows"
soup2=findInSoup(search)
key="Cited documents"
not_key="Uncited documents"
cited_doc, uncited_doc=ParseSoupTable(soup2, key, not_key)

In [146]:
cited_doc

{'Year': ['1999',
  '2000',
  '2001',
  '2002',
  '2003',
  '2004',
  '2005',
  '2006',
  '2007',
  '2008',
  '2009',
  '2010',
  '2011',
  '2012',
  '2013',
  '2014',
  '2015',
  '2016',
  '2017',
  '2018',
  '2019',
  '2020',
  '2021',
  '2022',
  '2023',
  '2024'],
 'Cited documents': ['42',
  '57',
  '61',
  '87',
  '101',
  '116',
  '136',
  '166',
  '187',
  '232',
  '283',
  '305',
  '325',
  '359',
  '389',
  '396',
  '392',
  '428',
  '460',
  '481',
  '448',
  '496',
  '505',
  '464',
  '384',
  '316'],
 'Description': "Ratio of a journal's items, grouped in three years windows, that have been cited at least once vs. those not cited during the following year."}

## Estimated financial value

In [147]:
search="It represents the potential financial worth of a journal. It is obtained by multiplying"
soup2=findInSoup(search)
key="Est. value (USD)"
not_key=""
fin_value=ParseSoupTable(soup2, key, not_key, finance=True)

In [148]:
fin_value

{'Year': ['1999',
  '2000',
  '2001',
  '2002',
  '2003',
  '2004',
  '2005',
  '2006',
  '2007',
  '2008',
  '2009',
  '2010',
  '2011',
  '2012',
  '2013',
  '2014',
  '2015',
  '2016',
  '2017',
  '2018',
  '2019',
  '2020',
  '2021',
  '2022',
  '2023',
  '2024'],
 'Est. value (USD)': ['215976',
  '447683',
  '630411',
  '836130',
  '1164270',
  '1300962',
  '1552066',
  '1957793',
  '2665567',
  '2898335',
  '3269659',
  '3602842',
  '4303532',
  '4114127',
  '4537639',
  '4734497',
  '4892505',
  '4745563',
  '5083274',
  '5057992',
  '5199992',
  '5228703',
  '4838286',
  '4550087',
  '4143497',
  '3568361'],
 'Description': "It represents the potential financial worth of a journal. It is obtained by multiplying the journal's Estimated APC by the total number of citable documents published over the past five years. This value reflects the hypothetical revenue a journal could generate based on its estimated publication costs and scholarly output."}